# 🤗 HuggingFace & Transformers — Practical NLP Tutorial

Welcome! This notebook walks you through the **HuggingFace ecosystem** step by step:

| Section | What you'll learn |
|---|---|
| 1 · Setup | Installing libraries, HF Hub basics |
| 2 · Pipeline API | Zero-effort inference in one line |
| 3 · AutoTokenizer | Turning text into tensors |
| 4 · AutoModel + AutoClass | Full control over the model |
| 5 · Fine-tuning loop | Training on your own data with `Trainer` |
| 6 · Bonus — Computer Vision | Image classification with ViT |

> **Prerequisites:** Python 3.8+, basic familiarity with PyTorch tensors.


## 1 · Installation & Imports

In [ ]:
# Install the core HuggingFace libraries (run once)
# transformers  – model architectures + weights
# datasets      – easy access to thousands of public datasets
# evaluate      – standardised metrics (accuracy, F1, …)
# accelerate    – hardware-agnostic training (CPU / GPU / TPU)
!pip install -q transformers datasets evaluate accelerate


In [ ]:
import torch
from transformers import (
    pipeline,                      # high-level inference API
    AutoTokenizer,                 # tokeniser that matches any checkpoint
    AutoModelForSequenceClassification,   # auto-selects architecture for the task
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer,
)
from datasets import load_dataset
import evaluate

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# HuggingFace Hub lets you browse, search and download models
# https://huggingface.co/models  ← explore thousands of free checkpoints


## 2 · The `pipeline` API — Instant Inference

`pipeline()` is the **easiest entry point** into Transformers.  
It wraps tokenisation → model forward pass → post-processing into a single callable.

Under the hood it:
1. Downloads a pretrained model + tokeniser from the Hub (cached after first run).  
2. Moves everything to the right device.  
3. Returns human-readable results.


In [ ]:
# ── 2a. Sentiment Analysis ──────────────────────────────────────────────────
# Default checkpoint for "text-classification": distilbert-base-uncased-finetuned-sst-2-english
sentiment = pipeline("text-classification")

texts = [
    "I absolutely loved this film — a masterpiece!",
    "The service was dreadful and the food was cold.",
    "It was an okay experience, nothing special.",
]

results = sentiment(texts)

for text, res in zip(texts, results):
    label = res["label"]
    score = res["score"]
    print(f"[{label:>8s}  {score:.2%}]  {text}")


In [ ]:
# ── 2b. Zero-Shot Classification ───────────────────────────────────────────
# No fine-tuning needed — just provide candidate labels at runtime.
# Model: facebook/bart-large-mnli (trained on Natural Language Inference)
zsc = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

headline = "NASA astronauts complete six-hour spacewalk to repair solar panel"
candidate_labels = ["space exploration", "politics", "sports", "technology", "arts"]

result = zsc(headline, candidate_labels=candidate_labels)

print(f"Text  : {result['sequence']}
")
for label, score in zip(result["labels"], result["scores"]):
    bar = "█" * int(score * 30)
    print(f"  {label:<20s} {score:.2%}  {bar}")


In [ ]:
# ── 2c. Other popular pipeline tasks ───────────────────────────────────────
# Just swap the task string — same API for everything.

tasks = [
    ("summarization",         "This is a text summarisation pipeline. You give it a long document "
                              "and it returns a shorter version preserving the key information."),
    ("translation_en_to_fr",  "HuggingFace makes it easy to translate between languages."),
    ("fill-mask",             "The capital of France is [MASK]."),  # BERT-style masked LM
]

for task, text in tasks:
    print(f"▶ Task: {task}")
    pipe = pipeline(task)
    out  = pipe(text, max_length=50) if task == "summarization" else pipe(text)
    # `out` may be a list of dicts depending on the task
    print("  →", out)
    print()


## 3 · Tokenisation with `AutoTokenizer`

Before a model can process text it must be converted into **token IDs** — integers
that index into the model's vocabulary.  Every pretrained checkpoint ships with its
own tokeniser; `AutoTokenizer` selects the right one automatically.

Key concepts:
- **Vocabulary** — the set of all known tokens (often 30k – 50k).
- **Subword splitting** — unknown words are broken into smaller pieces (WordPiece / BPE).
- **Special tokens** — `[CLS]`, `[SEP]`, `<s>`, `</s>` etc. that the model was trained with.
- **Attention mask** — tells the model which positions are real tokens vs. padding.


In [ ]:
CHECKPOINT = "distilbert-base-uncased"  # small, fast, great for demos

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
print(f"Vocabulary size : {tokenizer.vocab_size:,}")
print(f"Special tokens  : {tokenizer.all_special_tokens}")


In [ ]:
sentence = "HuggingFace Transformers make NLP surprisingly simple!"

# --- Step-by-step tokenisation ---
tokens   = tokenizer.tokenize(sentence)          # string tokens (subwords)
ids      = tokenizer.convert_tokens_to_ids(tokens)
print("Tokens :", tokens)
print("IDs    :", ids)

# --- Production usage: call the tokeniser directly ---
# Returns a BatchEncoding (dict-like) with input_ids, attention_mask, etc.
encoded = tokenizer(
    sentence,
    padding=True,          # pad to longest sequence in the batch
    truncation=True,       # truncate if longer than model's max length
    max_length=128,
    return_tensors="pt",   # "pt"=PyTorch, "tf"=TensorFlow, "np"=NumPy
)

print("
Encoded keys   :", list(encoded.keys()))
print("input_ids shape:", encoded["input_ids"].shape)   # (batch, seq_len)
print("input_ids      :", encoded["input_ids"])
print("attention_mask :", encoded["attention_mask"])


In [ ]:
# --- Decoding: convert IDs back to text ---
decoded = tokenizer.decode(encoded["input_ids"][0], skip_special_tokens=True)
print("Decoded:", decoded)

# --- Batch tokenisation (padding in action) ---
batch = [
    "Short sentence.",
    "This is a slightly longer sentence that will need more tokens.",
    "Tiny.",
]
batch_enc = tokenizer(batch, padding=True, truncation=True, return_tensors="pt")
print("
Batch input_ids shape:", batch_enc["input_ids"].shape)
# All rows padded to the same length; attention_mask = 0 on padding positions
print(batch_enc["attention_mask"])


## 4 · `AutoModelForSequenceClassification` — Text Classification

`AutoModel*` classes load a pretrained model **and add a task-specific head** on top.
The head (e.g. a linear layer) converts the transformer's hidden states into logits
for your target classes.

HuggingFace provides an `Auto*` variant for every common NLP task:

| AutoClass | Task |
|---|---|
| `AutoModelForSequenceClassification` | document / sentence classification |
| `AutoModelForTokenClassification` | NER, POS tagging |
| `AutoModelForQuestionAnswering` | extractive QA |
| `AutoModelForCausalLM` | GPT-style text generation |
| `AutoModelForSeq2SeqLM` | translation, summarisation |


In [ ]:
# We'll use a checkpoint that's already fine-tuned on SST-2 sentiment
CHECKPOINT = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
model     = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT)

# Print a high-level summary of the architecture
print(model)
print(f"
Number of parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Label mapping       : {model.config.id2label}")


In [ ]:
# ── Manual forward pass ─────────────────────────────────────────────────────
sentence = "This tutorial is incredibly helpful and well-structured."

inputs = tokenizer(sentence, return_tensors="pt")

with torch.no_grad():          # no gradient computation needed for inference
    outputs = model(**inputs)  # **inputs unpacks input_ids + attention_mask

# outputs.logits shape: (batch_size, num_labels)
logits = outputs.logits
print("Raw logits :", logits)

# Convert logits → probabilities with softmax
probs = torch.softmax(logits, dim=-1)
pred_id = torch.argmax(probs, dim=-1).item()

print(f"Predicted class : {model.config.id2label[pred_id]}")
print(f"Confidence      : {probs[0][pred_id]:.2%}")


In [ ]:
# ── Batch inference with a DataLoader ──────────────────────────────────────
sentences = [
    "The acting was superb and the plot kept me on the edge of my seat.",
    "Boring, predictable, a complete waste of two hours.",
    "Some scenes were great but overall it felt disjointed.",
    "A triumph of storytelling. Easily the film of the decade.",
    "I fell asleep halfway through.",
]

inputs = tokenizer(sentences, padding=True, truncation=True, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

probs    = torch.softmax(outputs.logits, dim=-1)
pred_ids = torch.argmax(probs, dim=-1).tolist()

print(f"{'Sentiment':<12} {'Conf':>6}   Text")
print("-" * 70)
for pid, prob, text in zip(pred_ids, probs, sentences):
    label = model.config.id2label[pid]
    print(f"{label:<12} {prob[pid]:.1%}   {text}")


## 5 · Fine-tuning with `Trainer`

So far we've used **already fine-tuned** checkpoints.  
Now we'll fine-tune a base model on **our own dataset** using the `Trainer` API.

The workflow:
```
Raw text  →  AutoTokenizer  →  Tokenised dataset
                                      ↓
                         AutoModelForSequenceClassification
                                      ↓
                              TrainingArguments
                                      ↓
                                   Trainer.train()
                                      ↓
                                Trainer.evaluate()
```

We'll use the **IMDb** movie-review dataset (50 k reviews, binary sentiment).


In [ ]:
# Load dataset from the HuggingFace Hub — no manual downloading needed
dataset = load_dataset("imdb")
print(dataset)
print("
Example:")
print(dataset["train"][0])


In [ ]:
CHECKPOINT = "distilbert-base-uncased"   # base model (no task head yet)
tokenizer  = AutoTokenizer.from_pretrained(CHECKPOINT)

def tokenize_fn(examples):
    """Applied to the whole dataset in parallel batches."""
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=256,   # keep shorter for speed; original max is 512
    )

# Use a small subset for a quick demo — remove [:500] to train on full data
small_train = dataset["train"].shuffle(seed=42).select(range(500))
small_test  = dataset["test"].shuffle(seed=42).select(range(200))

tokenized_train = small_train.map(tokenize_fn, batched=True)
tokenized_test  = small_test.map(tokenize_fn,  batched=True)

print(tokenized_train)
print("
Features:", tokenized_train.features)


In [ ]:
# Load the base model and attach a 2-class classification head
model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT,
    num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1},
)

# ── Training hyperparameters ────────────────────────────────────────────────
# TrainingArguments holds every hyperparameter + hardware config.
training_args = TrainingArguments(
    output_dir="./results",          # where checkpoints are saved
    num_train_epochs=2,              # full passes over the training data
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,              # typical for transformer fine-tuning
    weight_decay=0.01,               # L2 regularisation
    eval_strategy="epoch",           # evaluate once per epoch
    save_strategy="epoch",
    load_best_model_at_end=True,     # restore best checkpoint after training
    logging_dir="./logs",
    logging_steps=20,
    report_to="none",                # disable W&B / MLflow for this demo
)

# ── Metric function ─────────────────────────────────────────────────────────
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return accuracy_metric.compute(predictions=preds, references=labels)


In [ ]:
# ── Trainer — ties everything together ─────────────────────────────────────
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# 🚀 Train!  (< 2 min on GPU, a few min on CPU with the small subset)
trainer.train()


In [ ]:
# ── Evaluate on the held-out test split ────────────────────────────────────
metrics = trainer.evaluate()
print(f"Test accuracy: {metrics['eval_accuracy']:.2%}")

# ── Save the model for later use ───────────────────────────────────────────
trainer.save_model("./my_sentiment_model")
tokenizer.save_pretrained("./my_sentiment_model")
print("
Model saved to ./my_sentiment_model")
print("Load it later with:")
print("  AutoTokenizer.from_pretrained('./my_sentiment_model')")
print("  AutoModelForSequenceClassification.from_pretrained('./my_sentiment_model')")


In [ ]:
# ── Quick inference with the fine-tuned model ──────────────────────────────
ft_pipeline = pipeline(
    "text-classification",
    model="./my_sentiment_model",
    tokenizer="./my_sentiment_model",
)

test_reviews = [
    "A stunning visual spectacle with a heart-pounding story.",
    "Terrible script, terrible pacing, terrible everything.",
]

for review in test_reviews:
    res = ft_pipeline(review)[0]
    print(f"[{res['label']:>8s}  {res['score']:.1%}]  {review}")


## 6 · Bonus — Computer Vision with `ViT`

The exact same HuggingFace API works for **vision models**.  
`google/vit-base-patch16-224` is a Vision Transformer (ViT) pre-trained on ImageNet-21k
and fine-tuned on ImageNet-1k — 1 000 object classes.

The pipeline handles:  
`PIL Image → feature extractor (resize, normalise) → ViT encoder → classification head → labels`


In [ ]:
from PIL import Image
import requests
from io import BytesIO
from transformers import AutoFeatureExtractor, AutoModelForImageClassification
import torch

# ── Load image from the web ─────────────────────────────────────────────────
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/43/Cute_dog.jpg/1280px-Cute_dog.jpg"
response = requests.get(url, timeout=10)
image = Image.open(BytesIO(response.content)).convert("RGB")
image   # Jupyter will display the image inline


In [ ]:
# ── Option A: pipeline (one-liner) ─────────────────────────────────────────
img_classifier = pipeline(
    "image-classification",
    model="google/vit-base-patch16-224",
)

predictions = img_classifier(image, top_k=5)

print("Top-5 predictions:")
for p in predictions:
    bar = "█" * int(p["score"] * 40)
    print(f"  {p['label']:<30s} {p['score']:.2%}  {bar}")


In [ ]:
# ── Option B: manual forward pass (more control) ───────────────────────────
CHECKPOINT = "google/vit-base-patch16-224"

feature_extractor = AutoFeatureExtractor.from_pretrained(CHECKPOINT)
vit_model         = AutoModelForImageClassification.from_pretrained(CHECKPOINT)

# feature_extractor resizes to 224×224 and normalises pixel values
inputs = feature_extractor(images=image, return_tensors="pt")
print("Pixel values shape:", inputs["pixel_values"].shape)
# Shape: (batch, channels, height, width) = (1, 3, 224, 224)

with torch.no_grad():
    outputs = vit_model(**inputs)

logits   = outputs.logits                     # (1, 1000)
top5_idx = torch.topk(logits, k=5).indices[0].tolist()

print("
Top-5 via manual forward pass:")
for idx in top5_idx:
    label = vit_model.config.id2label[idx]
    score = torch.softmax(logits, dim=-1)[0][idx].item()
    print(f"  [{idx:>4d}] {label:<40s} {score:.2%}")


In [ ]:
# ── Key ViT concepts ───────────────────────────────────────────────────────
#
#  Patch size 16 means the 224×224 image is split into (224/16)² = 196 patches.
#  Each patch is linearly projected to an embedding, treated like a "word token".
#  The transformer encoder then processes all 196 patch embeddings + 1 [CLS] token.
#  The [CLS] representation is passed to the classification head.
#
#  This is directly analogous to BERT on text — same architecture, different modality!

print("Image tokens (patches):", (224 // 16) ** 2, "+ 1 [CLS] =", (224 // 16) ** 2 + 1)


## 🎉 Summary & Next Steps

### What we covered

| Concept | Key class / function |
|---|---|
| One-liner inference | `pipeline("task")` |
| Tokenisation | `AutoTokenizer` |
| Sequence classification | `AutoModelForSequenceClassification` |
| Fine-tuning | `Trainer` + `TrainingArguments` |
| Image classification | `AutoModelForImageClassification` |

### Further reading

- 📖 [HuggingFace Course](https://huggingface.co/learn/nlp-course/) — free, interactive
- 🔍 [Model Hub](https://huggingface.co/models) — browse 500k+ checkpoints
- 📦 [Datasets Hub](https://huggingface.co/datasets) — 50k+ public datasets
- ⚡ [PEFT / LoRA](https://huggingface.co/docs/peft) — parameter-efficient fine-tuning

### Ideas to try next

1. Swap `"distilbert-base-uncased"` for `"roberta-base"` — same code, different model.
2. Try `"token-classification"` pipeline for Named Entity Recognition.
3. Fine-tune on a multi-class dataset (e.g. AG News: 4 classes).
4. Push your model to the Hub with `trainer.push_to_hub("your-username/my-model")`.
